# 02 — Baseline Models, Duration Leakage and Hyperparameter Tuning

This notebook rebuilds the baseline and tuning experiment directly from the source dataset. It separates pre-call modelling from the post-call `duration` diagnostic and performs every search from scratch with `SEED=42`.

No `data/`, `results/` or `figures/` folder is required and no cached parameter/result file is read.

## 1. Validation design and deployable feature sets

A stratified 80/20 development/final-holdout split is fixed with seed 42. Model selection happens only inside development: 24,000 rows for LR tuning, 8,000 for tuning validation, and a reproducible 10,000-row stratified HGB tuning subsample. The final 8,000-row holdout is untouched until selection is complete.

In [ ]:
from inspect import signature
import json,time,warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,ParameterGrid,ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score
warnings.filterwarnings("ignore")
SEED=42; DATA_SOURCE="term-deposit-marketing-2020-labelled.csv"
NUM_PRE=["age","balance","day","campaign"]; NUM_DURATION=["age","balance","day","duration","campaign"]; CAT=["job","marital","education","default","housing","loan","contact","month"]; PRE=NUM_PRE+CAT; WITH_DURATION=NUM_DURATION+CAT
df=pd.read_csv(DATA_SOURCE)
if "y_binary" in df.columns: df=df.drop(columns="y_binary")
df["y_binary"]=df["y"].eq("yes").astype(int); y=df["y_binary"].to_numpy(); idx=np.arange(len(df))
dev_idx,test_idx=train_test_split(idx,test_size=.20,stratify=y,random_state=SEED); train_idx,val_idx=train_test_split(dev_idx,test_size=.25,stratify=y[dev_idx],random_state=SEED); hgb_tune_idx,_=train_test_split(train_idx,train_size=10000,stratify=y[train_idx],random_state=SEED)
split_summary=pd.DataFrame({"split":["LR tuning train","tuning validation","HGB tuning subsample","development total","final untouched test"],"n":[len(train_idx),len(val_idx),len(hgb_tune_idx),len(dev_idx),len(test_idx)],"base_rate":[y[train_idx].mean(),y[val_idx].mean(),y[hgb_tune_idx].mean(),y[dev_idx].mean(),y[test_idx].mean()]}); display(split_summary.round(4))

## 2. Preprocessing, baseline models and metric definitions

Numeric predictors are standardised and categorical predictors are one-hot encoded. Default LR and HGB settings are held fixed across the pre-call and with-duration comparisons so the effect of `duration` is not confounded with a model-setting change.

In [ ]:
def prep(features): return ColumnTransformer([("num",StandardScaler(),[f for f in features if f in NUM_DURATION]),("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),[f for f in features if f in CAT])])
def lr_model(params):
    q=params.copy(); penalty=q.pop("penalty")
    if signature(LogisticRegression).parameters["penalty"].default=="deprecated": q["l1_ratio"]=1.0 if penalty=="l1" else 0.0
    else: q["penalty"]=penalty
    return LogisticRegression(solver="liblinear",max_iter=3000,random_state=SEED,**q)
def hgb_model(params): return HistGradientBoostingClassifier(random_state=SEED,**params)
def pipe(features,model): return Pipeline([("prep",prep(features)),("model",model)])
def baseline(kind):
    if kind=="LR": return LogisticRegression(C=1.0,penalty="l2",solver="lbfgs",max_iter=1000,tol=1e-4,class_weight=None,random_state=SEED)
    return HistGradientBoostingClassifier(loss="log_loss",learning_rate=.1,max_iter=100,max_leaf_nodes=31,max_depth=None,min_samples_leaf=20,l2_regularization=0,max_bins=255,early_stopping="auto",class_weight=None,random_state=SEED)
def metrics(yt,p,t=.5):
    z=(p>=t).astype(int); return {"accuracy":accuracy_score(yt,z),"precision":precision_score(yt,z,zero_division=0),"recall":recall_score(yt,z,zero_division=0),"f1":f1_score(yt,z,zero_division=0),"pr_auc":average_precision_score(yt,p),"roc_auc":roc_auc_score(yt,p)}

## 3. Hyperparameter search spaces and reproducibility

LR exhaustively evaluates 60 configurations per feature condition. HGB samples the same 100 candidates per condition from a 28,800-combination space because `ParameterSampler` uses `random_state=42`. Selection is by highest validation PR-AUC, with ROC-AUC as the deterministic tie-breaker. Across the four searches this is 320 candidate configurations.

In [ ]:
LR_GRID=list(ParameterGrid({"C":np.logspace(-3,2,15),"penalty":["l1","l2"],"class_weight":[None,"balanced"]}))
HGB_SPACE={"max_iter":[50,100,150,200,300,400],"max_depth":[3,4,5,6,8,None],"learning_rate":[.01,.03,.05,.1,.2],"max_leaf_nodes":[15,31,63,127],"l2_regularization":[0,.1,.5,1,2],"min_samples_leaf":[10,20,30,50],"class_weight":[None,"balanced"]}
HGB_SPACE_SIZE=int(np.prod([len(v) for v in HGB_SPACE.values()])); HGB_CANDIDATES=list(ParameterSampler(HGB_SPACE,n_iter=100,random_state=SEED)); assert len(LR_GRID)==60 and len(HGB_CANDIDATES)==100 and HGB_SPACE_SIZE==28800
def choose(rows): return pd.DataFrame(rows).sort_values(["validation_pr_auc","validation_roc_auc"],ascending=[False,False]).reset_index(drop=True)
def tune_lr(features):
    pp=prep(features); Xt=pp.fit_transform(df.loc[train_idx,features]); Xv=pp.transform(df.loc[val_idx,features]); rows=[]
    for run,params in enumerate(LR_GRID,1):
        m=lr_model(params); m.fit(Xt,y[train_idx]); p=m.predict_proba(Xv)[:,1]; rows.append({"run":run,"validation_pr_auc":average_precision_score(y[val_idx],p),"validation_roc_auc":roc_auc_score(y[val_idx],p),"params":json.dumps(params,sort_keys=True)})
    z=choose(rows); return json.loads(z.iloc[0].params),z
def tune_hgb(features):
    pp=prep(features); Xt=pp.fit_transform(df.loc[hgb_tune_idx,features]); Xv=pp.transform(df.loc[val_idx,features]); rows=[]
    for run,params in enumerate(HGB_CANDIDATES,1):
        q={**params,"early_stopping":True}; m=hgb_model(q); m.fit(Xt,y[hgb_tune_idx]); p=m.predict_proba(Xv)[:,1]; rows.append({"run":run,"validation_pr_auc":average_precision_score(y[val_idx],p),"validation_roc_auc":roc_auc_score(y[val_idx],p),"params":json.dumps(q,sort_keys=True)})
    z=choose(rows); return json.loads(z.iloc[0].params),z

## 4. Fresh tuning run and selected configurations

The following cell always reruns all four searches. It does not load selected-parameter JSON, candidate CSVs or prior outputs.

In [ ]:
selected={}; candidate_tables={}; audit=[]
for key,kind,features in [("LR_pre_call","LR",PRE),("LR_with_duration","LR",WITH_DURATION),("HGB_pre_call","HGB",PRE),("HGB_with_duration","HGB",WITH_DURATION)]:
    best,cands=tune_lr(features) if kind=="LR" else tune_hgb(features); selected[key]=best; candidate_tables[key]=cands; audit.append({"configuration":key,"model":kind,"candidate_configurations":len(cands),"best_validation_pr_auc":cands.iloc[0].validation_pr_auc,"best_validation_roc_auc":cands.iloc[0].validation_roc_auc})
audit=pd.DataFrame(audit); display(audit.round(6)); display(pd.DataFrame([{"configuration":k,"selected_parameters":json.dumps(v,sort_keys=True)} for k,v in selected.items()])); print("Total candidate configurations:",int(audit.candidate_configurations.sum()))

## 5. Untouched-holdout baseline and tuned evaluation

After selection, each model is refitted on the full 80% development set and evaluated once on the untouched 20% holdout. This cleanly separates the effect of `duration`, tuning and model family.

In [ ]:
def evaluate(features,model):
    est=pipe(features,model); est.fit(df.loc[dev_idx,features],y[dev_idx]); p=est.predict_proba(df.loc[test_idx,features])[:,1]; return metrics(y[test_idx],p)
rows=[]
for kind,features,label,key in [("LR",PRE,"Pre-call","LR_pre_call"),("LR",WITH_DURATION,"With duration","LR_with_duration"),("HGB",PRE,"Pre-call","HGB_pre_call"),("HGB",WITH_DURATION,"With duration","HGB_with_duration")]:
    rows.append({"stage":"Untuned/default","model":kind,"feature_set":label,**evaluate(features,baseline(kind))}); tuned=lr_model(selected[key]) if kind=="LR" else hgb_model(selected[key]); rows.append({"stage":"Tuned","model":kind,"feature_set":label,**evaluate(features,tuned)})
holdout=pd.DataFrame(rows); display(holdout.round(6))

## 6. Duration leakage and tuning deltas

`duration` is classified as leakage because of timing: it is known only after the call. The performance increase below is supporting evidence, not the definition of leakage. Tuning deltas are shown separately so baseline and tuned results are never conflated.

In [ ]:
duration_rows=[]; tuning_rows=[]
for stage in ["Untuned/default","Tuned"]:
    d=holdout[holdout.stage.eq(stage)]
    for kind in ["LR","HGB"]:
        a=d[(d.model==kind)&(d.feature_set=="Pre-call")].iloc[0]; b=d[(d.model==kind)&(d.feature_set=="With duration")].iloc[0]; duration_rows.append({"stage":stage,"model":kind,"precall_pr_auc":a.pr_auc,"duration_pr_auc":b.pr_auc,"pr_auc_change":b.pr_auc-a.pr_auc,"precall_roc_auc":a.roc_auc,"duration_roc_auc":b.roc_auc,"roc_auc_change":b.roc_auc-a.roc_auc})
for fs in ["Pre-call","With duration"]:
    for kind in ["LR","HGB"]:
        a=holdout[(holdout.stage=="Untuned/default")&(holdout.model==kind)&(holdout.feature_set==fs)].iloc[0]; b=holdout[(holdout.stage=="Tuned")&(holdout.model==kind)&(holdout.feature_set==fs)].iloc[0]; tuning_rows.append({"model":kind,"feature_set":fs,"pr_auc_untuned":a.pr_auc,"pr_auc_tuned":b.pr_auc,"pr_auc_change":b.pr_auc-a.pr_auc,"roc_auc_untuned":a.roc_auc,"roc_auc_tuned":b.roc_auc,"roc_auc_change":b.roc_auc-a.roc_auc})
display(pd.DataFrame(duration_rows).round(6)); display(pd.DataFrame(tuning_rows).round(6))

## 7. Six-metric comparison panel

The untouched-holdout table is paired with the required 2×3 Accuracy / Precision / Recall / F1 / PR-AUC / ROC-AUC panel. Duration-inclusive configurations are clearly labelled as diagnostic.

In [ ]:
metrics6=["accuracy","precision","recall","f1","pr_auc","roc_auc"]; plot_df=holdout.copy(); plot_df["configuration"]=plot_df.model+" | "+plot_df.stage.replace({"Untuned/default":"Default"})+" | "+plot_df.feature_set.replace({"With duration":"+ duration"})
fig,axes=plt.subplots(2,3,figsize=(16,9)); axes=axes.ravel()
for ax,m in zip(axes,metrics6): ax.bar(range(len(plot_df)),plot_df[m]); ax.set_title(m.replace("_"," ").upper()); ax.set_ylim(0,1); ax.set_xticks(range(len(plot_df))); ax.set_xticklabels(plot_df.configuration,rotation=55,ha="right",fontsize=8)
fig.suptitle("Untouched-Holdout Model Comparison",y=1.01); plt.tight_layout(); plt.show()

## 8. Notebook-02 conclusion

Notebook 02 is the canonical source for selected hyperparameters and baseline/tuned holdout metrics. Downstream notebooks transfer only the freshly selected **pre-call** settings; they do not copy old cached values or reintroduce `duration`.